In [8]:
import pandas as pd
import yaml

intermediate_dir = 'output/intermediate/'
cfg = yaml.safe_load(open('configs/settings.yaml'))

In [55]:
# load CHAS HUD data
chas = (
    pd.read_excel('data/' + cfg['data']['hud_chas'], sheet_name='Data')
    .rename(columns={
        'Income by Cost Burden': 'income_by_cost_burden',
        'Cost burden > 30%  (Renters only)': 'cost_burden_over_30',
        'Total (Renters and Owners)': 'total_hh'
        })
)

chas['county_id'] = chas['County'].map(cfg['county_map'])


In [56]:
# Calculate total households by county from CHAS data
chas_tot_hh = chas.groupby('county_id')['total_hh'].sum()

# Use vacancy rates to calculate current OFM household counts
vac = pd.read_csv(intermediate_dir + '/vacancy_rates.csv').set_index('county_id')
ofm_units = pd.read_csv(intermediate_dir + 'ofm_housing_estimates.csv').set_index('county_id')
ofm_hh = round(ofm_units['total_housing_units'] * (1 - vac['vacancy_rate']), 0)

# calculate adjustment ratio by county to apply to CHAS data to grow to current OFM HH
adj_ratio = ofm_hh / chas_tot_hh


In [58]:
chas['adj_ratio'] = chas['county_id'].map(adj_ratio)

In [60]:
chas['cost_burden_over_30_adj'] = round(chas['cost_burden_over_30'] * chas['adj_ratio'], 0)

In [61]:
chas.groupby('county_id')['cost_burden_over_30'].sum()

county_id
53033    169335
53035     14890
53053     57510
53061     46055
Name: cost_burden_over_30, dtype: int64

In [62]:
chas.groupby('county_id')['cost_burden_over_30_adj'].sum()

county_id
53033    185623.0
53035     15827.0
53053     60620.0
53061     49303.0
Name: cost_burden_over_30_adj, dtype: float64

In [63]:
chas

,County,income_by_cost_burden,cost_burden_over_30,total_hh,county_id,adj_ratio,cost_burden_over_30_adj
0,King County,Household Income <= 30% HAMFI,61145,114475,53033,1.096188,67026.0
1,King County,Household Income >30% to <=50% HAMFI,47510,91390,53033,1.096188,52080.0
2,King County,Household Income >50% to <=80% HAMFI,39510,117325,53033,1.096188,43310.0
3,King County,Household Income >80% to <=100% HAMFI,12285,84505,53033,1.096188,13467.0
4,King County,Household Income >100% HAMFI,8885,508575,53033,1.096188,9740.0
5,Kitsap County,Household Income <= 30% HAMFI,4855,10770,53035,1.063017,5161.0
6,Kitsap County,Household Income >30% to <=50% HAMFI,3685,9360,53035,1.063017,3917.0
7,Kitsap County,Household Income >50% to <=80% HAMFI,4450,18615,53035,1.063017,4730.0
8,Kitsap County,Household Income >80% to <=100% HAMFI,1325,12090,53035,1.063017,1408.0
9,Kitsap County,Household Income >100% HAMFI,575,55200,53035,1.063017,611.0
